# 03 — NYC scaling

When we scale the pipeline up from DTBK (~4 km², 667 POIs) to NYC Metro
(~785 km², 57 k POIs), the LoRA-Llama becomes the **best Top-1 and Top-5**
predictor of any of the 9 methods we evaluated.

This notebook shows the headline table and the 4-panel + 3D residual
diagnostic figures that ship in the poster.


## 1. The 9-method comparison (2022 OOS, 500k visits)

In [ ]:
import json, pandas as pd
m = json.load(open('../results/nyc_metrics.json'))
df = pd.DataFrame(m['methods']).set_index('name')
def bold_max_min(s, op):
    target = op(s)
    return [f'**{v:.4f}**' if v == target else f'{v:.4f}' for v in s]
out = df.copy()
for col in ['top_1','top_5','spearman','ndcg_10','ndcg_50']:
    out[col] = bold_max_min(df[col], max)
out['js'] = bold_max_min(df['js'], min)
out

**Read as**: the four popularity/distance-only models collapse to popularity
(β → 0) because the candidate sampler has already pre-filtered by distance.
Among rich-feature models, the LoRA-Llama posts the best Top-1 and Top-5;
the LightGBM ranker dominates aggregate ranking metrics.

## 2. The 4-panel actual-vs-predicted scatter

In [ ]:
from IPython.display import Image
Image('../figures/nyc/temporal_scatter_4panel.png')

## 3. The 3D residual demand surface

In [ ]:
Image('../figures/nyc/oos_3d_demand_surface.png')

**3D residual** = predicted minus actual visit mass at every NYC location.
Red areas are over-predicted, blue are under-predicted. Notice the model
slightly under-predicts mid-Brooklyn and over-predicts Manhattan's
financial district — both consistent with sparser Cuebiq coverage in
gentrifying neighborhoods.

## 4. Spatial hexbin maps

In [ ]:
Image('../figures/nyc/poster_temporal_status_spatial_map.png')